## Libraries & Organizing

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_BATCHES =  PROJECT_ROOT / "data" / "batches"
DATA_OUTPUT =  PROJECT_ROOT / "data" / "output"

## Datasets

In [ ]:
# ============================================================
# Load datasets
# ============================================================

dataset_llama = pd.read_csv(DATA_PROCESSED / "final_dataset_NAs.csv")
dataset_qwen = pd.read_csv(DATA_PROCESSED / "final_dataset_NAs_qwen.csv")


In [ ]:
dataset_llama.columns

In [ ]:
dataset_llama[['forward_looking_intensity', 'dict_score']].corr()

In [ ]:
dataset_qwen[['forward_looking_intensity', 'dict_score']].corr()

## Regression

In [ ]:

# ============================================================
# Settings
# ============================================================

car_vars = [
    "car_immediate",
    "car_medium",
    "car_long"
]

fe_specs = {
    "FIRM": "C(permno)",
    "IND": "C(gind)"
}

output_dir = DATA_PROCESSED / "regressions"
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# Group rare qualitative LLM categories
# ============================================================

qual_cols = [
    "main_focus",
    "managerial_horizon",
]

def group_rare_categories(df, cols, min_count=30):
    df = df.copy()

    for col in cols:
        df[col] = df[col].astype(str).str.strip()

        counts = df[col].value_counts()
        keep_categories = counts[counts >= min_count].index

        df[col + "_grouped"] = df[col].where(
            df[col].isin(keep_categories),
            "Other"
        )

    return df


dataset_llama = group_rare_categories(
    dataset_llama,
    qual_cols,
    min_count=30
)

dataset_qwen = group_rare_categories(
    dataset_qwen,
    qual_cols,
    min_count=30
)

datasets = {
    "llama": dataset_llama,
    "qwen": dataset_qwen
}


In [ ]:



# ============================================================
# Clean regression output
# ============================================================

def clean_regression_output(model, title):
    params = model.params
    bse = model.bse
    pvals = model.pvalues

    rows = []

    for var in params.index:
        # Hide firm/industry FE only
        if var.startswith("C(permno)") or var.startswith("C(gind)"):
            continue

        rows.append({
            "Variable": var,
            "Coef.": params[var],
            "Std.Err.": bse[var],
            "P>|t|": pvals[var]
        })

    out = pd.DataFrame(rows)

    text = []
    text.append(title)
    text.append("-" * 80)
    text.append(out.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    text.append("")
    text.append(f"N: {int(model.nobs)}")
    text.append(f"R-squared: {model.rsquared:.4f}")
    text.append(f"Adj. R-squared: {model.rsquared_adj:.4f}")
    text.append("")
    text.append("Note: Firm/industry fixed effects are included where specified but not reported.")
    text.append("Year effects and grouped qualitative LLM categories are reported.")
    text.append("Standard errors are clustered by firm (PERMNO).")

    return "\n".join(text)


# ============================================================
# Regression function
# ============================================================
def run_regressions(df, model_name, car_var, fe_name, fe_term):
    df = df.copy()

    # Contextualized FLI
    df["context_score"] = df[
        [
            "specificity",
            "economic_substance",
            "certainty"
        ]
    ].mean(axis=1)

    df["contextual_fli"] = (
        df["forward_looking_intensity"]
        * df["context_score"]
    )

    # ========================================================
    # Specification 1: Contextualized FLI + fixed effects only
    # ========================================================

    vars_ctx_fe_only = [
        car_var,
        "contextual_fli",
        "tone",
        "permno",
        "year",
        "gind"
    ]

    reg_ctx_fe_only = df[vars_ctx_fe_only].dropna().copy()

    formula_ctx_fe_only = f"""
    {car_var} ~
    contextual_fli +
    tone +
    C(year) +
    {fe_term}
    """

    model_ctx_fe_only = smf.ols(
        formula=formula_ctx_fe_only,
        data=reg_ctx_fe_only
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_ctx_fe_only["permno"]}
    )

    # ========================================================
    # Specification 2: Contextualized FLI + controls + fixed effects
    # ========================================================

    vars_ctx_controls = [
        car_var,
        "contextual_fli",
        "tone",
        "bm",
        "log_word_count",
        "log_assets",
        "permno",
        "year",
        "gind"
    ]

    reg_ctx_controls = df[vars_ctx_controls].dropna().copy()

    formula_ctx_controls = f"""
    {car_var} ~
    contextual_fli +
    tone +
    bm +
    log_word_count +
    log_assets +
    C(year) +
    {fe_term}
    """

    model_ctx_controls = smf.ols(
        formula=formula_ctx_controls,
        data=reg_ctx_controls
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_ctx_controls["permno"]}
    )

    # ========================================================
    # Specification 3: Dictionary FLI + controls + fixed effects
    # ========================================================

    vars_dict_controls = [
        car_var,
        "dict_score",
        "bm",
        "log_word_count",
        "log_assets",
        "permno",
        "year",
        "gind"
    ]

    reg_dict_controls = df[vars_dict_controls].dropna().copy()

    formula_dict_controls = f"""
    {car_var} ~
    dict_score +
    bm +
    log_word_count +
    log_assets +
    C(year) +
    {fe_term}
    """

    model_dict_controls = smf.ols(
        formula=formula_dict_controls,
        data=reg_dict_controls
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_dict_controls["permno"]}
    )

    # ========================================================
    # Specification 4: Contextualized FLI + dictionary FLI + controls + fixed effects
    # ========================================================

    vars_ctx_dict_controls = [
        car_var,
        "contextual_fli",
        "tone",
        "dict_score",
        "bm",
        "log_word_count",
        "log_assets",
        "permno",
        "year",
        "gind",
        "main_focus_grouped",
        "managerial_horizon_grouped"
    ]

    reg_ctx_dict_controls = df[vars_ctx_dict_controls].dropna().copy()

    formula_ctx_dict_controls = f"""
    {car_var} ~
    contextual_fli +
    dict_score +
    tone +
    bm +
    log_word_count +
    log_assets +
    C(year) +
    C(main_focus_grouped) +
    C(managerial_horizon_grouped) +
    {fe_term}
    """

    model_ctx_dict_controls = smf.ols(
        formula=formula_ctx_dict_controls,
        data=reg_ctx_dict_controls
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": reg_ctx_dict_controls["permno"]}
    )

    # ========================================================
    # Save output
    # ========================================================

    file_name = f"contextual_regression_{car_var}_{fe_name}_{model_name}.txt"
    file_path = output_dir / file_name

    with open(file_path, "w") as f:
        f.write(f"Model: {model_name.upper()}\n")
        f.write(f"Dependent variable: {car_var}\n")
        f.write(f"Fixed effects: {fe_name}\n")
        f.write("=" * 80 + "\n\n")

        f.write(clean_regression_output(
            model_ctx_fe_only,
            "SPECIFICATION 1: CONTEXTUALIZED FLI + FIXED EFFECTS ONLY"
        ))

        f.write("\n\n" + "=" * 80 + "\n\n")

        f.write(clean_regression_output(
            model_ctx_controls,
            "SPECIFICATION 2: CONTEXTUALIZED FLI + CONTROLS"
        ))

        f.write("\n\n" + "=" * 80 + "\n\n")

        f.write(clean_regression_output(
            model_dict_controls,
            "SPECIFICATION 3: DICTIONARY FLI + CONTROLS"
        ))

        f.write("\n\n" + "=" * 80 + "\n\n")

        f.write(clean_regression_output(
            model_ctx_dict_controls,
            "SPECIFICATION 4: CONTEXTUALIZED FLI + DICTIONARY FLI + CONTROLS"
        ))

    return (
        model_ctx_fe_only,
        model_ctx_controls,
        model_dict_controls,
        model_ctx_dict_controls
    )
# ============================================================
# Run all regressions
# ============================================================

results = {}

for model_name, df in datasets.items():
    for car_var in car_vars:
        for fe_name, fe_term in fe_specs.items():

            print(f"Running {model_name} | {car_var} | {fe_name}")

            (
                ctx_fe_only_model,
                ctx_controls_model,
                dict_controls_model,
                ctx_dict_controls_model
            ) = run_regressions(
                df=df,
                model_name=model_name,
                car_var=car_var,
                fe_name=fe_name,
                fe_term=fe_term
            )

            results[(model_name, car_var, fe_name, "ctx_fe_only")] = ctx_fe_only_model
            results[(model_name, car_var, fe_name, "ctx_controls")] = ctx_controls_model
            results[(model_name, car_var, fe_name, "dict_controls")] = dict_controls_model
            results[(model_name, car_var, fe_name, "ctx_dict_controls")] = ctx_dict_controls_model

print("Done. Regression files saved in:", output_dir)


In [ ]:
# ============================================================
# LaTeX regression table export
# Keeps the old regression block untouched
# ============================================================

from matplotlib.pyplot import table

from statsmodels.iolib.summary2 import summary_col

latex_output_dir = DATA_PROCESSED / "regression_tables_latex"
latex_output_dir.mkdir(parents=True, exist_ok=True)


def make_latex_regression_table(models, car_var, fe_name, model_name):
    """
    Creates one LaTeX regression table with five specifications:
    (1) LLM FLI with controls only
    (2) LLM FLI and contextual dimensions with controls and fixed effects
    (3) Additive LLM dimensions with dictionary FLI, without qualitative variables
    (4) Additive LLM dimensions with grouped qualitative variables
    (5) Contextualized FLI with grouped qualitative variables
    """

    info_dict = {
        "N": lambda x: f"{int(x.nobs)}",
        "R$^2$": lambda x: f"{x.rsquared:.3f}",
        "Adj. R$^2$": lambda x: f"{x.rsquared_adj:.3f}"
    }

    table = summary_col(
        models,
        stars=True,
        float_format="%0.4f",
        model_names=["(1)", "(2)", "(3)", "(4)"],
        info_dict=info_dict
    )

    df_table = table.tables[0].copy()

    # --------------------------------------------------------
    # Always keep numerical/core variables
    # --------------------------------------------------------

    always_keep = [
        "Intercept",
        "forward_looking_intensity",
        "contextual_fli",
        "dict_score",
        "specificity",
        "economic_substance",
        "tone",
        "certainty",
        "bm",
        "log_word_count",
        "log_assets",
        "C(year)"
    ]

    # --------------------------------------------------------
    # Detect significant categorical variables
    # --------------------------------------------------------

    significant_categoricals = set()

    for model in models:

        for var, pval in model.pvalues.items():

            is_categorical = (
                var.startswith("C(main_focus_grouped)") or
                var.startswith("C(managerial_horizon_grouped)")
            )

            if is_categorical and pval < 0.10:
                significant_categoricals.add(var)

    # --------------------------------------------------------
    # Final keep logic
    # --------------------------------------------------------

    keep_rows = []

    i = 0

    while i < len(df_table.index):

        idx = str(df_table.index[i])

        keep = (
            any(idx.startswith(prefix) for prefix in always_keep)
            or idx in significant_categoricals
        )

        if keep:

            keep_rows.append(i)

            # Keep SE row
            if i + 1 < len(df_table.index):

                next_idx = str(df_table.index[i + 1])

                if next_idx.strip() == "":
                    keep_rows.append(i + 1)

            i += 2

        else:
            i += 1

    table.tables[0] = df_table.iloc[keep_rows].copy()

    # --------------------------------------------------------
    # Re-add model statistics rows
    # --------------------------------------------------------

    stats_rows = [
        "N",
        "R$^2$",
        "Adj. R$^2$"
    ]

    stats_df = df_table.loc[
        [idx for idx in df_table.index if idx in stats_rows]
    ].copy()

    table.tables[0] = pd.concat(
        [table.tables[0], stats_df]
    )

    # --------------------------------------------------------
    # Rename variables for thesis-style table
    # --------------------------------------------------------

    rename_dict = {
        "Intercept": "Constant",
        "forward_looking_intensity": "LLM FLI",
        "contextual_fli": "Contextualized FLI",
        "dict_score": "Dictionary FLI",
        "specificity": "Specificity",
        "economic_substance": "Economic Substance",
        "tone": "Tone",
        "certainty": "Certainty",
        "bm": "Book-to-Market",
        "log_word_count": "Log Word Count",
        "log_assets": "Log Assets"
    }

    table.tables[0].rename(index=rename_dict, inplace=True)

    # --------------------------------------------------------
    # Add specification indicator rows
    # --------------------------------------------------------

    fe_rows = pd.DataFrame(
        {
            "(1)": ["Yes", "No",  fe_name, "No",  "Yes"],
            "(2)": ["Yes", "Yes", fe_name, "No",  "Yes"],
            "(3)": ["Yes", "Yes", fe_name, "Yes", "No"],
            "(4)": ["Yes", "Yes", fe_name, "Yes", "Yes"]
        },
        index=[
            "Year FE",
            "Controls",
            "Fixed Effects",
            "Dictionary FLI Included",
            "Contextualized FLI Included"
        ]
    )

    table.tables[0] = pd.concat([table.tables[0], fe_rows])

    # --------------------------------------------------------
    # Convert to LaTeX
    # --------------------------------------------------------

    latex = table.as_latex()
    
    latex = latex.replace(
    "Adj. R\\$\\textasciicircum \\{2\\}\\$",
    "Adj. R\\$^2\\$ \\\\\n\\midrule"
)

    caption = (
        f"Condensed regression results for "
        f"{car_var.replace('_', ' ')} "
        f"using {model_name.capitalize()} "
        f"with {fe_name.lower()} fixed effects"
)

    label = f"tab:reg_{car_var}_{fe_name.lower()}_{model_name}"

    latex = latex.replace(
        "\\begin{table}",
        "\\begin{table}[htbp]\n\\centering"
    )

    latex = latex.replace(
        "\\caption{}",
        f"\\caption{{{caption}}}\n\\label{{{label}}}"
    )

    latex = latex.replace("\\end{table}\n","")

    latex += (
        "\n\\vspace{0.2cm}\n"
        "\\begin{minipage}{0.95\\textwidth}\n"
        "\\footnotesize\n"
        "\\textit{Notes:} This table reports OLS regression results. "
        "Standard errors are reported in parentheses and clustered by firm (PERMNO). "
        "Firm or industry fixed effects are included where indicated but not reported. "
        "Grouped qualitative LLM categories are reported where included. "
        "$^{*}p<0.10$, $^{**}p<0.05$, $^{***}p<0.01$.\n"
        "\\end{minipage}\n"
        "\\end{table}\n"
    )

    file_name = f"contextual_regression_condensed_{car_var}_{fe_name}_{model_name}.tex"
    file_path = latex_output_dir / file_name

    with open(file_path, "w") as f:
        f.write(latex)

    return latex, file_path


# ============================================================
# Export LaTeX tables from existing results dictionary
# ============================================================

for model_name in ["llama", "qwen"]:
    for car_var in car_vars:
        for fe_name in ["FIRM", "IND"]:

            models = [
                results[(model_name, car_var, fe_name, "ctx_fe_only")],
                results[(model_name, car_var, fe_name, "ctx_controls")],
                results[(model_name, car_var, fe_name, "dict_controls")],
                results[(model_name, car_var, fe_name, "ctx_dict_controls")]
            ]

            latex, path = make_latex_regression_table(
                models=models,
                car_var=car_var,
                fe_name=fe_name,
                model_name=model_name
            )

            print("Saved:", path)